<a href="https://colab.research.google.com/github/LuciaKajanova/dspracticum25_flowers_team/blob/homework_appartments_part_1/appartments_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Appartments in Prague - price prediction

In [17]:
# Import

import pandas as pd
import numpy as np

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.linear_model import LinearRegression

# Load the data
train = pd.read_csv("appartments_train.csv")
test  = pd.read_csv("appartments_test.csv")

print(train.shape, test.shape)
train.head()


(5000, 32) (1020, 32)


,id,address,layout,price,gps_lat,gps_lon,area,floor,total_floors,construction,...,poi_school_kindergarten_nearest,poi_transport_count,poi_transport_nearest,poi_grocery_count,poi_grocery_nearest,poi_restaurant_count,poi_restaurant_nearest,text,first_seen,last_seen
0,6980,"Tavolníková, Praha - Krč",2+kk,6450000.0,50.019823,14.461106,57.0,5,NaN,Panelová,...,290.0,5,189.0,5,160.0,5,344.0,Nabízím k prodeji byt 2+kk/45 m2 s chodbou pře...,2025-08-20,2025-08-23
1,1860,"Pitterova, Praha 3 - Žižkov",3+kk,14750000.0,50.085597,14.468936,93.0,2,12.0,Smíšená,...,211.0,5,70.0,5,324.0,5,225.0,Nabízíme k prodeji světlý a prostorný byt 3+kk...,2025-06-27,2025-07-08
2,5225,"Perucká, Praha 2 - Vinohrady",1+kk,5490000.0,50.068807,14.437938,27.0,2,4.0,Cihlová,...,179.0,5,176.0,5,377.0,5,152.0,"Prodej bytu 1+kk o velikosti 27m², umístěného ...",2025-07-16,2025-08-12
3,4441,"Brožíkova, Praha - Košíře",1+1,6352000.0,50.071383,14.388532,38.0,2,NaN,Cihlová,...,220.0,5,84.0,5,46.0,5,110.0,Přímý vlastník nabízí k prodeji zrekonstruovan...,2025-07-01,2025-07-31
4,7143,"Hnězdenská, Praha 8 - Troja",2+kk,6190000.0,50.126809,14.423477,46.0,11,12.0,Smíšená,...,258.0,5,178.0,5,121.0,5,153.0,Připravujeme k prodeji byt 2 + kk po rekonstru...,2025-08-22,2025-09-01


In [18]:
#  Custom transformer for total_floors

# Fills in missing total_floors with the median within the 'construction' group.
# Assumes columns 'construction' and 'total_floors'.

class TotalFloorsByConstruction(BaseEstimator, TransformerMixin):
    def __init__(self, construction_col="construction", floors_col="total_floors"):
        self.construction_col = construction_col
        self.floors_col = floors_col
        self.medians_ = None
        self.global_median_ = None

    def fit(self, X, y=None):
        X_df = X.copy()
        self.medians_ = (
            X_df.groupby(self.construction_col)[self.floors_col]
                .median()
                .to_dict()
        )
        self.global_median_ = X_df[self.floors_col].median()
        return self

    def transform(self, X):
        X = X.copy()

        def fill_row(row):
            val = row[self.floors_col]
            if pd.isna(val):
                cons = row[self.construction_col]
                return self.medians_.get(cons, self.global_median_)
            return val

        mask_na = X[self.floors_col].isna()
        if mask_na.any():
            X.loc[mask_na, self.floors_col] = X.loc[mask_na].apply(fill_row, axis=1)

        return X


In [19]:
# Preparation of X, y, and column types

# Target
y = train["price"]

# Columns we don't want in model
drop_cols = ["price", "id", "address", "text", "first_seen", "last_seen"]

X = train.drop(columns=drop_cols)
X_test_final = test.drop(columns=["id", "address", "text", "first_seen", "last_seen"])

# 1) Logical NA → 0
logical_zero_cols = ["garden_area", "balcony_area", "cellar_area", "parking"]

for col in logical_zero_cols:
    X[col] = X[col].fillna(0)
    X_test_final[col] = X_test_final[col].fillna(0)

# 2) parking must be categorical → cast to string
X["parking"] = X["parking"].astype(str)
X_test_final["parking"] = X_test_final["parking"].astype(str)

# 3) Determine numeric & categorical columns again
numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [col for col in numeric_cols if col not in logical_zero_cols]

categorical_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()

if "parking" not in categorical_cols:
    categorical_cols.append("parking")

print("Numeric columns:", numeric_cols)
print("Categorical columns:", categorical_cols)


Numeric columns: ['gps_lat', 'gps_lon', 'area', 'floor', 'total_floors', 'poi_doctors_count', 'poi_doctors_nearest', 'poi_leisure_time_count', 'poi_leisure_time_nearest', 'poi_school_kindergarten_count', 'poi_school_kindergarten_nearest', 'poi_transport_count', 'poi_transport_nearest', 'poi_grocery_count', 'poi_grocery_nearest', 'poi_restaurant_count', 'poi_restaurant_nearest']
Categorical columns: ['layout', 'construction', 'condition', 'ownership', 'elevator', 'parking']


In [20]:
# Preprocessing and model pipelines

# 1) Numeric columns: NA -> median
numeric_transformer = SimpleImputer(strategy="median")

# 2) Categorical columns: NA -> "Unknown" + OneHotEncoder
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

# 3) ColumnTransformer combining numeric and categorical preprocessing
preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols),
    ]
)

# 4) Candidate models to compare
candidate_models = {
    "RandomForest": RandomForestRegressor(
        n_estimators=300,
        max_depth=None,
        random_state=42,
        n_jobs=-1,
    ),
    "HistGradientBoosting": HistGradientBoostingRegressor(
        max_depth=None,
        learning_rate=0.1,
        max_iter=300,
        random_state=42,
    ),
    "GradientBoosting": GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,
        random_state=42,
    ),
    "LinearRegression": LinearRegression(),
}

# 5) Build one Pipeline per model (same preprocessing, different final estimator)
pipelines = {}

for name, model in candidate_models.items():
    pipelines[name] = Pipeline(steps=[
        ("group_imputer", TotalFloorsByConstruction(
            construction_col="construction",
            floors_col="total_floors",
        )),
        ("preprocess", preprocess),
        ("model", model),
    ])

print("Pipelines ready for models:", list(pipelines.keys()))


Pipelines ready for models: ['RandomForest', 'HistGradientBoosting', 'GradientBoosting', 'LinearRegression']


In [21]:
# Train/valid split, training, and MAPE for all models

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

results = {}

for name, pipe in pipelines.items():
    print(f"\n=== Training model: {name} ===")
    pipe.fit(X_train, y_train)
    y_val_pred = pipe.predict(X_val)
    mape_val = mean_absolute_percentage_error(y_val, y_val_pred) * 100
    results[name] = mape_val
    print(f"Validation MAPE ({name}): {mape_val:.2f} %")

print("\n=== Summary (lower MAPE is better) ===")
for name, mape_val in results.items():
    print(f"{name}: {mape_val:.2f} %")

# Optionally: pick the best model name
best_model_name = min(results, key=results.get)
print(f"\nBest model on validation: {best_model_name} with MAPE {results[best_model_name]:.2f} %")




=== Training model: RandomForest ===
Validation MAPE (RandomForest): 11.94 %

=== Training model: HistGradientBoosting ===
Validation MAPE (HistGradientBoosting): 11.12 %

=== Training model: GradientBoosting ===
Validation MAPE (GradientBoosting): 11.70 %

=== Training model: LinearRegression ===
Validation MAPE (LinearRegression): 15.10 %

=== Summary (lower MAPE is better) ===
RandomForest: 11.94 %
HistGradientBoosting: 11.12 %
GradientBoosting: 11.70 %
LinearRegression: 15.10 %

Best model on validation: HistGradientBoosting with MAPE 11.12 %


In [22]:
# Training the best model on the full train set + prediction on test + saving submission

# Use the best model selected on validation
best_pipe = pipelines[best_model_name]
print(f"Fitting best model on full data: {best_model_name}")

best_pipe.fit(X, y)

test_pred = best_pipe.predict(X_test_final)

submission = pd.DataFrame({
    "id": test["id"],
    "price": test_pred
})

submission.to_csv("submission.csv", index=False)
submission.head()


Fitting best model on full data: HistGradientBoosting


,id,price
0,8795,7.487951e+06
1,6516,8.743566e+06
2,4714,5.433389e+06
3,8423,6.873613e+06
4,5361,7.864107e+06
